# sabi tutorial: 2-D banana benchmark

Walks through a sabi run end-to-end on a small benchmark whose 2-D geometry makes every step easy to visualize.

What we'll do:

1. Construct the validated `banana_2d()` `BenchmarkProblem` and visualize its target log-posterior.
2. Compose an `Algorithm`: a TinyGP emulator, Expected Improvement acquisition, and an MMD-vs-reference metric.
3. Run the sequential loop for a handful of rounds.
4. Visualize the resulting surrogate posterior, the points the loop chose, and how the MMD evolved.
5. Compare against a random baseline.

All cell outputs were cleared before commit. Re-execute in order.

## 1. Setup

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from probpipe import sample, unnormalized_log_prob

from sabi.acquisitions.ei import ExpectedImprovement
from sabi.acquisitions.random import PriorSampling
from sabi.algorithms import Algorithm, run
from sabi.emulators import TinyGPEmulator
from sabi.metrics.mmd import MMD
from sabi.problems.benchmarks import banana_2d

jax.config.update("jax_enable_x64", True)

## 2. The benchmark

`banana_2d()` returns a frozen `BenchmarkProblem` — fixed parameters, validated reference distribution. `Problem` is the flexible cousin if you want to vary `(d, a, b, c)` for development; benchmarks bake those in.

In [ ]:
bp = banana_2d()
print(f"name:             {bp.name}")
print(f"artifact_version: {bp.artifact_version}")
print(f"input_shape:      {bp.input_shape}")
print(f"reference samples: {bp.reference_distribution.samples.shape}")

Visualize the unnormalized target log-density on a grid, with reference samples overlaid.

In [ ]:
def grid_eval(target_fn, x_lo=-4.0, x_hi=4.0, y_lo=-10.0, y_hi=4.0, n=120):
    xs = jnp.linspace(x_lo, x_hi, n)
    ys = jnp.linspace(y_lo, y_hi, n)
    grid = jnp.stack(jnp.meshgrid(xs, ys, indexing="ij"), axis=-1).reshape(-1, 2)
    log_p = jnp.asarray(target_fn(grid)).reshape(n, n)
    return np.asarray(xs), np.asarray(ys), np.asarray(log_p)

xs, ys, log_p = grid_eval(bp.target_map)
ref_samples = np.asarray(bp.reference_distribution.samples)

fig, ax = plt.subplots(figsize=(5.5, 6))
cs = ax.contour(xs, ys, log_p.T, levels=20, cmap="viridis")
ax.scatter(ref_samples[:, 0], ref_samples[:, 1], s=2, alpha=0.15, color="black", label="reference")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("banana_2d: $\\log p(x)$ contours + reference samples")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Compose the algorithm

The `Algorithm` dataclass bundles every component the loop needs. Each field is independently swappable — that's how ablations are expressed in sabi.

We use:

- `TinyGPEmulator` — a simple GP surrogate over the parameter space.
- `ExpectedImprovement` — picks the next point where the emulator predicts the largest expected gain in log-density.
- `MMD` — measures distance between samples from the surrogate posterior and the benchmark's reference samples.

In [ ]:
def make_algorithm(acquisition):
    return Algorithm(
        emulator_factory=lambda: TinyGPEmulator(input_shape=(2,)),
        acquisition=acquisition,
        n_initial=8,
        n_rounds=12,
        metrics=(MMD(n_estimate_samples=512, n_reference_samples=512),),
    )

algo_ei = make_algorithm(ExpectedImprovement())
algo_rand = make_algorithm(PriorSampling())

## 4. Run the loop

`run(problem, algorithm, key)` executes the sequential loop end-to-end and returns a `RunResult` with the design points (`X`), raw target evaluations (`Y_raw`), the fitted emulator, per-round metric rows, and a final posterior estimate.

In [ ]:
key = jax.random.key(0)
result_ei = run(bp, algo_ei, key)
result_rand = run(bp, algo_rand, key)

print("EI    per-round mmd²:")
for i, row in enumerate(result_ei.per_round_metrics):
    print(f"  round {i:2d}: {row.get('mmd2', float('nan')):.4f}")
print(f"EI final mmd²:    {result_ei.final_metrics.get('mmd2', float('nan')):.4f}")
print(f"random final mmd²: {result_rand.final_metrics.get('mmd2', float('nan')):.4f}")

## 5. Visualize the surrogate posterior

The estimator on the run result is the **expected target** distribution — log-posterior built from the emulator's predictive mean composed with the problem's `LogDensityForm`. Contour-plot it and overlay the points the loop selected.

In [ ]:
def grid_estimate_log_density(estimate, n=120):
    xs = jnp.linspace(-4.0, 4.0, n)
    ys = jnp.linspace(-10.0, 4.0, n)
    grid = jnp.stack(jnp.meshgrid(xs, ys, indexing="ij"), axis=-1).reshape(-1, 2)
    log_p = jnp.asarray(unnormalized_log_prob(estimate, grid)).reshape(n, n)
    return np.asarray(xs), np.asarray(ys), np.asarray(log_p)

xs_e, ys_e, log_est = grid_estimate_log_density(result_ei.final_estimate)
X_design = np.asarray(result_ei.X)
X_initial = X_design[: algo_ei.n_initial]
X_acquired = X_design[algo_ei.n_initial:]

fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharex=True, sharey=True)
axes[0].contour(xs, ys, log_p.T, levels=20, cmap="viridis")
axes[0].set_title("target $\\log p(x)$")
axes[1].contour(xs_e, ys_e, log_est.T, levels=20, cmap="viridis")
axes[1].scatter(X_initial[:, 0], X_initial[:, 1], s=40, c="tab:blue", label="initial", edgecolor="white")
axes[1].scatter(X_acquired[:, 0], X_acquired[:, 1], s=40, c="tab:red", label="EI-acquired", edgecolor="white")
axes[1].set_title("surrogate $\\log \\hat p(x)$ (expected target) + design")
axes[1].legend(loc="lower right")
for ax in axes:
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
plt.tight_layout()
plt.show()

Or sample from the surrogate posterior directly (this calls NUTS under the hood via ProbPipe `condition_on`):

In [ ]:
key_sample = jax.random.key(42)
est_samples = np.asarray(
    sample(result_ei.final_estimate, key=key_sample, sample_shape=(1024,))
)

fig, ax = plt.subplots(figsize=(5.5, 6))
ax.contour(xs, ys, log_p.T, levels=20, cmap="viridis", alpha=0.3)
ax.scatter(est_samples[:, 0], est_samples[:, 1], s=4, alpha=0.3, color="tab:red", label="surrogate samples")
ax.scatter(ref_samples[:1024, 0], ref_samples[:1024, 1], s=4, alpha=0.3, color="black", label="reference")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("surrogate samples vs reference")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 6. Random vs EI: MMD over rounds

Each `per_round_metrics` row carries the metric values evaluated against the surrogate posterior at the end of that round. Plotting the MMD² curve is the simplest ablation comparison.

In [ ]:
def mmd2_curve(result):
    return [row.get("mmd2", float("nan")) for row in result.per_round_metrics]

ei_curve = mmd2_curve(result_ei)
rand_curve = mmd2_curve(result_rand)
rounds = np.arange(len(ei_curve))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rounds, ei_curve, marker="o", label="EI")
ax.plot(rounds, rand_curve, marker="s", label="random (prior sampling)")
ax.set_xlabel("round")
ax.set_ylabel("MMD$^2$ vs reference")
ax.set_title("banana_2d: posterior MMD by round")
ax.legend()
plt.tight_layout()
plt.show()

## Where to next

- Swap the emulator (`TinyGPEmulator` → `DSPGPEmulator` from `sabi.emulators.gpjax`) and rerun.
- Swap the acquisition (`ExpectedImprovement` → its `ContinuousMultiStartOptimizer` variant for gradient-based search).
- Try the moderate-d cousin: `from sabi.problems.benchmarks import banana_10d`. The same `Algorithm` composes; only the visualizations need adjusting.
- See `docs/design.md` for the full conceptual layering, and `docs/notation.md` for the shape conventions used throughout.